# LeetCode 543 — Diameter of Binary Tree bằng Python

**Link đề:** https://leetcode.com/problems/diameter-of-binary-tree/
**Độ khó:** Easy · **Chủ đề:** Tree, Depth-First Search, Binary Tree, DP on Trees

```python
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def diameterOfBinaryTree(self, root: Optional[TreeNode]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu **đường kính** (diameter) của cây: đường đi dài nhất (tính bằng **số cạnh**) giữa hai node bất kỳ.
2. Ôn `TreeNode`, dựng cây từ list level order, **đệ quy** và **chiều cao** (height/depth).
3. Nhận ra mấu chốt: mọi đường đi đều có một **node cao nhất** (đỉnh), và độ dài qua đỉnh đó = `depth(left) + depth(right)`.
4. Đi từ baseline `O(n²)` (tính chiều cao lại nhiều lần) lên **một lần DFS hậu thứ tự** `O(n)` — kỹ thuật "trả về một thứ, cập nhật một thứ khác".
5. Viết thêm bản lặp bằng stack (tránh giới hạn đệ quy), trace call stack, chứng minh đúng và test đối chiếu.

## 1. Đề bài

Cho `root` của một cây nhị phân, trả về **độ dài đường kính** của cây.

- **Đường kính** = độ dài đường đi dài nhất giữa **hai node bất kỳ**. Đường này **có thể không đi qua root**.
- **Độ dài** đường đi = **số cạnh** (edges) trên đường, không phải số node.

```text
        1
       / \
      2   3          Đường dài nhất: 4 - 2 - 1 - 3  (hoặc 5 - 2 - 1 - 3)
     / \             -> 3 cạnh -> đáp án 3
    4   5
```

| `root` (level order) | Output |
|---|---:|
| `[1,2,3,4,5]` | 3 |
| `[1,2]` | 1 |
| `[1]` | 0 |

**Constraints:** số node trong `[1, 10^4]`; `-100 <= Node.val <= 100`.

Lưu ý: cây có thể lệch hẳn một phía (như một "linked list") sâu tới `10^4` tầng → cần để ý giới hạn đệ quy của Python.

## 2. Nền tảng: `TreeNode`, dựng cây và thuật ngữ

- **Node** có `val`, `left`, `right` (con trái/phải hoặc `None`).
- **Cạnh** (edge): nối một node với con của nó. Cây `n` node có `n - 1` cạnh.
- **Lá** (leaf): node không có con.
- **Chiều cao/độ sâu** của một cây con (trong notebook này): số **node** trên đường dài nhất từ gốc cây con xuống một lá. Cây rỗng có chiều cao 0.

Ta tự định nghĩa `TreeNode` để notebook chạy độc lập.

In [1]:
from typing import Optional
from collections import deque

class TreeNode:
    def __init__(self, val: int = 0, left: "Optional[TreeNode]" = None, right: "Optional[TreeNode]" = None):
        self.val = val
        self.left = left
        self.right = right

root = TreeNode(1, TreeNode(2, TreeNode(4), TreeNode(5)), TreeNode(3))
print(root.val, root.left.val, root.right.val, root.left.left.val, root.left.right.val)

1 2 3 4 5


### 2.1. Dựng cây từ list level order

LeetCode ghi cây theo **tầng**, trái sang phải; `null` (Python `None`) là vị trí trống. Ta dùng **queue** `collections.deque`: `append` thêm vào cuối, `popleft` lấy ra từ đầu (vào trước ra trước — FIFO). Node nào được lấy ra thì gán hai con tiếp theo trong list cho nó.

In [2]:
def build_tree(values: list) -> Optional[TreeNode]:
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root

def draw(node: Optional[TreeNode], depth: int = 0) -> None:
    # In cây xoay 90 độ: nhánh phải ở trên, nhánh trái ở dưới
    if node is None:
        return
    draw(node.right, depth + 1)
    print("    " * depth + str(node.val))
    draw(node.left, depth + 1)

draw(build_tree([1, 2, 3, 4, 5]))

    3
1
        5
    2
        4


### 2.2. Đệ quy tính chiều cao

Hàm đệ quy trên cây gồm:
1. **Base case**: `node is None` → trả 0.
2. **Bước đệ quy**: chiều cao = `1 + max(chiều cao trái, chiều cao phải)`.

`max(a, b)` trả về số lớn hơn.

In [3]:
def height(node: Optional[TreeNode]) -> int:
    if node is None:
        return 0
    return 1 + max(height(node.left), height(node.right))

t = build_tree([1, 2, 3, 4, 5])
print("height(root) =", height(t))          # 3 node: 1-2-4
print("height(node 2) =", height(t.left))   # 2 node: 2-4
print("height(node 3) =", height(t.right))  # 1
print("height(None) =", height(None))       # 0

height(root) = 3
height(node 2) = 2
height(node 3) = 1
height(None) = 0


## 3. Quan sát then chốt: mọi đường đi đều có một "đỉnh"

Lấy một đường đi bất kỳ giữa hai node `u` và `v`. Trên đường đó có đúng **một node cao nhất** (gần root nhất) — gọi là **đỉnh** `p`. Đường đi gồm:
- một nhánh đi **xuống bên trái** của `p`,
- và một nhánh đi **xuống bên phải** của `p` (một trong hai có thể rỗng).

Muốn đường qua `p` **dài nhất** thì mỗi nhánh phải đi xuống **sâu nhất có thể**:

> **Đường dài nhất có đỉnh tại `p`** = `height(p.left) + height(p.right)` (số cạnh).

Vì sao là số cạnh? Nhánh trái có `height(p.left)` node, mỗi node đó nối với cha của nó bằng 1 cạnh, tính cả cạnh nối với `p` → đúng `height(p.left)` cạnh. Tương tự bên phải.

Đáp án = **max** của giá trị này trên **mọi node** `p` (vì đỉnh có thể là bất kỳ node nào, không nhất thiết root).

In [4]:
t = build_tree([1, 2, 3, 4, 5])
for name, p in [("1", t), ("2", t.left), ("3", t.right), ("4", t.left.left)]:
    print(f"đỉnh {name}: height(left)={height(p.left)}, height(right)={height(p.right)} -> {height(p.left) + height(p.right)} cạnh")

đỉnh 1: height(left)=2, height(right)=1 -> 3 cạnh
đỉnh 2: height(left)=1, height(right)=1 -> 2 cạnh
đỉnh 3: height(left)=0, height(right)=0 -> 0 cạnh
đỉnh 4: height(left)=0, height(right)=0 -> 0 cạnh


### Phản ví dụ: đường kính không qua root

Cây dưới đây có nhánh trái rất "béo", nhánh phải chỉ một node. Đường dài nhất nằm hoàn toàn trong cây con trái.

```text
            1
           / \
          2   3
         / \
        4   5
       /     \
      6       7
     /         \
    8           9
```

Qua root: `height(2) + height(3) = 4 + 1 = 5`. Qua node 2: `height(4) + height(5) = 3 + 3 = 6` → đáp án 6 (đường `8-6-4-2-5-7-9`).

In [5]:
skew = build_tree([1, 2, 3, 4, 5, None, None, 6, None, None, 7, 8, None, None, 9])
draw(skew)
print("qua root:", height(skew.left) + height(skew.right))
print("qua node 2:", height(skew.left.left) + height(skew.left.right))

    3
1
                9
            7
        5
    2
        4
            6
                8
qua root: 5
qua node 2: 6


## 4. Cách 1 — Baseline: tại mỗi node gọi `height` hai lần

Dịch thẳng quan sát ở mục 3:
1. Duyệt mọi node `p` (bằng đệ quy).
2. Tính `height(p.left) + height(p.right)`.
3. Lấy max.

In [6]:
def diameter_naive(root: Optional[TreeNode]) -> int:
    if root is None:
        return 0
    through_root = height(root.left) + height(root.right)
    best_left = diameter_naive(root.left)
    best_right = diameter_naive(root.right)
    return max(through_root, best_left, best_right)

print(diameter_naive(build_tree([1, 2, 3, 4, 5])))  # 3
print(diameter_naive(build_tree([1, 2])))           # 1
print(diameter_naive(skew))                         # 6

3
1
6


### Nút thắt của baseline

`height` được gọi lại cho cùng một cây con **rất nhiều lần**: node ở độ sâu `d` bị tính chiều cao bởi tất cả `d` tổ tiên của nó.

- Cây cân bằng: `O(n log n)`.
- Cây lệch (như linked list): `1 + 2 + ... + n` = **`O(n²)`** → với `n = 10^4` khoảng 5·10^7 bước, quá chậm trong Python.

Ta đếm số lần gọi `height` để thấy rõ.

In [7]:
calls = 0
def height_counted(node):
    global calls
    calls += 1
    if node is None:
        return 0
    return 1 + max(height_counted(node.left), height_counted(node.right))

def diameter_naive_counted(root):
    if root is None:
        return 0
    through = height_counted(root.left) + height_counted(root.right)
    return max(through, diameter_naive_counted(root.left), diameter_naive_counted(root.right))

def build_chain(n: int) -> Optional[TreeNode]:
    # Cây lệch trái: 1 -> 2 -> ... -> n
    root = None
    for v in range(n, 0, -1):
        root = TreeNode(v, root, None)
    return root

for n in [10, 100, 400]:
    calls = 0
    d = diameter_naive_counted(build_chain(n))
    print(f"chain n={n:>3}: diameter={d}, số lần gọi height = {calls}")

chain n= 10: diameter=9, số lần gọi height = 110
chain n=100: diameter=99, số lần gọi height = 10100
chain n=400: diameter=399, số lần gọi height = 160400


Gọi `global calls` để hàm được phép **sửa** biến `calls` ở ngoài (nếu không, Python coi `calls` là biến cục bộ). Số lần gọi tăng **theo bình phương** `n`.

## 5. Insight — Gộp hai việc vào **một lần duyệt hậu thứ tự**

Để tính `height(p)`, ta **đã phải** tính `height(p.left)` và `height(p.right)`. Đó cũng chính là hai số cần cho "đường dài nhất qua đỉnh `p`"!

⇒ Viết **một** hàm `depth(node)`:
- **Trả về** chiều cao của `node` (để cha dùng).
- **Tiện tay cập nhật** biến `best = max(best, left + right)` (đáp án toàn cục).

Thứ tự "tính con trước, xử lý cha sau" gọi là duyệt **hậu thứ tự** (post-order). Mỗi node chỉ được thăm **một lần** ⇒ `O(n)`.

Mẫu tư duy này rất hay gặp: **"hàm đệ quy trả về một thông tin cho cha, còn đáp án thật được gom vào biến ngoài"**.

### 5.1. Biến ngoài trong hàm lồng nhau: `nonlocal`

Khi định nghĩa hàm bên trong hàm, muốn **gán lại** biến của hàm ngoài phải khai báo `nonlocal`. (Đọc thì không cần; gán thì cần.)

In [8]:
def outer():
    best = 0
    def update(x):
        nonlocal best
        best = max(best, x)
    for x in [3, 7, 2]:
        update(x)
    return best

print(outer())   # 7

7


In [9]:
def outer_wrong():
    best = 0
    def update(x):
        best = max(best, x)   # thiếu nonlocal -> Python coi best là biến cục bộ chưa gán
    update(5)
    return best

try:
    outer_wrong()
except UnboundLocalError as e:
    print("UnboundLocalError:", e)

UnboundLocalError: cannot access local variable 'best' where it is not associated with a value


## 6. Cách 2 — DFS một lần (lời giải chính)

Ghép từng đoạn:

```python
def depth(node):
    if node is None:          # base case: cây rỗng cao 0
        return 0
    left = depth(node.left)   # chiều cao cây con trái
    right = depth(node.right) # chiều cao cây con phải
    best = max(best, left + right)   # đường dài nhất có đỉnh ở node
    return 1 + max(left, right)      # chiều cao của node, trả cho cha
```

Đáp án là `best` sau khi gọi `depth(root)`.

In [10]:
def diameter_dfs(root: Optional[TreeNode]) -> int:
    best = 0

    def depth(node: Optional[TreeNode]) -> int:
        nonlocal best
        if node is None:
            return 0
        left = depth(node.left)
        right = depth(node.right)
        best = max(best, left + right)
        return 1 + max(left, right)

    depth(root)
    return best

print(diameter_dfs(build_tree([1, 2, 3, 4, 5])))  # 3
print(diameter_dfs(build_tree([1, 2])))           # 1
print(diameter_dfs(build_tree([1])))              # 0
print(diameter_dfs(skew))                         # 6

3
1
0
6


## 7. Trace call stack

Cây `[1,2,3,4,5]`. Thứ tự hoàn thành (post-order): 4, 5, 2, 3, 1.

| Node xong | `left` | `right` | `left+right` | `best` sau | trả về cho cha |
|---:|---:|---:|---:|---:|---:|
| 4 | 0 | 0 | 0 | 0 | 1 |
| 5 | 0 | 0 | 0 | 0 | 1 |
| 2 | 1 | 1 | 2 | 2 | 2 |
| 3 | 0 | 0 | 0 | 2 | 1 |
| 1 | 2 | 1 | **3** | **3** | 3 |

Đáp án `best = 3`. Hàm trace dưới đây in thụt lề theo độ sâu lời gọi để thấy call stack.

In [11]:
def trace_diameter(root: Optional[TreeNode]) -> int:
    best = 0

    def depth(node, level):
        nonlocal best
        pad = "  " * level
        if node is None:
            return 0
        print(f"{pad}vào node {node.val}")
        left = depth(node.left, level + 1)
        right = depth(node.right, level + 1)
        old = best
        best = max(best, left + right)
        print(f"{pad}xong node {node.val}: left={left}, right={right}, qua đỉnh={left + right}, best {old}->{best}, trả {1 + max(left, right)}")
        return 1 + max(left, right)

    depth(root, 0)
    print("Đường kính =", best)
    return best

trace_diameter(build_tree([1, 2, 3, 4, 5]))

vào node 1
  vào node 2
    vào node 4
    xong node 4: left=0, right=0, qua đỉnh=0, best 0->0, trả 1
    vào node 5
    xong node 5: left=0, right=0, qua đỉnh=0, best 0->0, trả 1
  xong node 2: left=1, right=1, qua đỉnh=2, best 0->2, trả 2
  vào node 3
  xong node 3: left=0, right=0, qua đỉnh=0, best 2->2, trả 1
xong node 1: left=2, right=1, qua đỉnh=3, best 2->3, trả 3
Đường kính = 3


3

In [12]:
# Trace ví dụ đường kính không qua root
trace_diameter(skew)

vào node 1
  vào node 2
    vào node 4
      vào node 6
        vào node 8
        xong node 8: left=0, right=0, qua đỉnh=0, best 0->0, trả 1
      xong node 6: left=1, right=0, qua đỉnh=1, best 0->1, trả 2
    xong node 4: left=2, right=0, qua đỉnh=2, best 1->2, trả 3
    vào node 5
      vào node 7
        vào node 9
        xong node 9: left=0, right=0, qua đỉnh=0, best 2->2, trả 1
      xong node 7: left=0, right=1, qua đỉnh=1, best 2->2, trả 2
    xong node 5: left=0, right=2, qua đỉnh=2, best 2->2, trả 3
  xong node 2: left=3, right=3, qua đỉnh=6, best 2->6, trả 4
  vào node 3
  xong node 3: left=0, right=0, qua đỉnh=0, best 6->6, trả 1
xong node 1: left=4, right=1, qua đỉnh=5, best 6->6, trả 5
Đường kính = 6


6

Ở cây `skew`: tại node 2, `left=3, right=3` → `best = 6`. Lên tới root: `left=4, right=1` → chỉ 5 < 6, `best` giữ nguyên 6. Đúng như dự đoán ở mục 3.

## 8. Code nộp LeetCode — DFS đệ quy

Trên LeetCode, `TreeNode` và `Optional` đã có sẵn. Dùng thuộc tính `self.best` cũng là một cách thay cho `nonlocal`.

In [13]:
class Solution:
    def diameterOfBinaryTree(self, root: Optional[TreeNode]) -> int:
        best = 0

        def depth(node: Optional[TreeNode]) -> int:
            nonlocal best
            if node is None:
                return 0
            left = depth(node.left)
            right = depth(node.right)
            best = max(best, left + right)
            return 1 + max(left, right)

        depth(root)
        return best

sol = Solution()
print(sol.diameterOfBinaryTree(build_tree([1, 2, 3, 4, 5])))
print(sol.diameterOfBinaryTree(build_tree([1, 2])))

3
1


### Lưu ý về giới hạn đệ quy

Python mặc định giới hạn khoảng 1000 tầng gọi hàm (`sys.getrecursionlimit()`). Cây lệch `10^4` node sẽ vượt → `RecursionError`. Hai cách xử lý:
1. Tăng giới hạn: `sys.setrecursionlimit(20000)` (thường đủ trên LeetCode).
2. Viết bản **lặp** dùng stack (mục 9).

In [14]:
import sys
print("Giới hạn đệ quy hiện tại:", sys.getrecursionlimit())
chain = build_chain(20000)
try:
    diameter_dfs(chain)
except RecursionError:
    print("Cây lệch 20000 node -> RecursionError với đệ quy mặc định")

Giới hạn đệ quy hiện tại: 1000
Cây lệch 20000 node -> RecursionError với đệ quy mặc định


## 9. Cách 3 — DFS lặp bằng stack (post-order thủ công)

Ý tưởng: mô phỏng post-order bằng stack chứa cặp `(node, visited)`:
- Lần đầu gặp node (`visited=False`): đẩy lại `(node, True)`, rồi đẩy hai con. Như vậy hai con được xử lý **trước**.
- Lần thứ hai (`visited=True`): hai con đã xong, đọc chiều cao của chúng từ dict `h`, tính `best` và `h[node]`.

`h` là **dict** ánh xạ node → chiều cao. `h.get(key, 0)` trả về 0 nếu key không có (dùng cho con `None`).

In [15]:
h = {}
h["a"] = 3
print(h.get("a", 0), h.get("b", 0))   # 3 0

3 0


In [16]:
def diameter_iterative(root: Optional[TreeNode]) -> int:
    if root is None:
        return 0
    best = 0
    h = {None: 0}                      # chiều cao của cây rỗng
    stack = [(root, False)]
    while stack:
        node, visited = stack.pop()
        if visited:
            left, right = h[node.left], h[node.right]
            best = max(best, left + right)
            h[node] = 1 + max(left, right)
        else:
            stack.append((node, True))
            if node.right:
                stack.append((node.right, False))
            if node.left:
                stack.append((node.left, False))
    return best

print(diameter_iterative(build_tree([1, 2, 3, 4, 5])))  # 3
print(diameter_iterative(skew))                         # 6
print(diameter_iterative(build_chain(10000)))           # 9999, không RecursionError

3
6
9999


In [17]:
def trace_iterative(root):
    best, h = 0, {None: 0}
    stack = [(root, False)]
    while stack:
        node, visited = stack.pop()
        if visited:
            l, r = h[node.left], h[node.right]
            best = max(best, l + r)
            h[node] = 1 + max(l, r)
            print(f"xử lý {node.val}: l={l}, r={r}, best={best}, h={h[node]}")
        else:
            stack.append((node, True))
            for child in (node.right, node.left):
                if child:
                    stack.append((child, False))
            print(f"mở {node.val}: stack = {[(n.val, v) for n, v in stack]}")
    return best

trace_iterative(build_tree([1, 2, 3, 4, 5]))

mở 1: stack = [(1, True), (3, False), (2, False)]
mở 2: stack = [(1, True), (3, False), (2, True), (5, False), (4, False)]
mở 4: stack = [(1, True), (3, False), (2, True), (5, False), (4, True)]
xử lý 4: l=0, r=0, best=0, h=1
mở 5: stack = [(1, True), (3, False), (2, True), (5, True)]
xử lý 5: l=0, r=0, best=0, h=1
xử lý 2: l=1, r=1, best=2, h=2
mở 3: stack = [(1, True), (3, True)]
xử lý 3: l=0, r=0, best=2, h=1
xử lý 1: l=2, r=1, best=3, h=3


3

### Code nộp LeetCode — bản lặp

In [18]:
class Solution:
    def diameterOfBinaryTree(self, root: Optional[TreeNode]) -> int:
        best = 0
        h = {None: 0}
        stack = [(root, False)] if root else []
        while stack:
            node, visited = stack.pop()
            if visited:
                left, right = h[node.left], h[node.right]
                best = max(best, left + right)
                h[node] = 1 + max(left, right)
            else:
                stack.append((node, True))
                if node.right:
                    stack.append((node.right, False))
                if node.left:
                    stack.append((node.left, False))
        return best

print(Solution().diameterOfBinaryTree(build_tree([1, 2, 3, 4, 5])))

3


## 10. Cách 4 (tham khảo) — Hai lần BFS trên đồ thị vô hướng

Một kỹ thuật kinh điển cho **cây tổng quát**: coi cây là đồ thị vô hướng.
1. BFS từ một node bất kỳ, tìm node **xa nhất** `u`.
2. BFS từ `u`, khoảng cách xa nhất chính là đường kính.

Vì sao đúng (ý chính): node xa nhất từ một điểm bất kỳ luôn là **một đầu mút** của một đường kính nào đó. Cách này cần dựng danh sách kề (cha ↔ con) nên tốn thêm bộ nhớ, nhưng rất hữu ích khi gặp bài "đường kính của cây" cho dưới dạng danh sách cạnh.

In [19]:
def diameter_two_bfs(root: Optional[TreeNode]) -> int:
    if root is None:
        return 0
    adj = {}                                  # danh sách kề: node -> list hàng xóm
    stack = [root]
    while stack:
        node = stack.pop()
        adj.setdefault(node, [])
        for child in (node.left, node.right):
            if child:
                adj[node].append(child)
                adj.setdefault(child, []).append(node)
                stack.append(child)

    def farthest(start):
        dist = {start: 0}
        q = deque([start])
        last = start
        while q:
            x = q.popleft()
            last = x
            for y in adj[x]:
                if y not in dist:
                    dist[y] = dist[x] + 1
                    q.append(y)
        return last, dist[last]

    u, _ = farthest(root)
    _, d = farthest(u)
    return d

print(diameter_two_bfs(build_tree([1, 2, 3, 4, 5])), diameter_two_bfs(skew))

3 6


`dict.setdefault(key, default)`: nếu `key` chưa có thì gán `default` rồi trả về giá trị đó; nếu có rồi thì trả giá trị cũ. Node cuối cùng lấy ra khỏi queue BFS là một node xa nhất (BFS duyệt theo khoảng cách tăng dần).

## 11. Vì sao Cách 2 đúng?

**Invariant:** khi `depth(node)` trả về,
1. giá trị trả về đúng bằng chiều cao (số node) của cây con gốc `node`;
2. `best` ≥ độ dài mọi đường đi **nằm trọn** trong cây con gốc `node`, và `best` luôn bằng độ dài của một đường đi có thật.

Chứng minh quy nạp theo cây con:
- `node is None`: trả 0, không có đường đi nào ✔.
- Giả sử đúng cho hai con. Mọi đường đi trong cây con `node` hoặc nằm trọn trong cây con trái, hoặc trọn trong cây con phải (đã được `best` phủ), hoặc **có đỉnh là `node`** — dài nhất là `left + right`, và ta vừa `max` vào `best` ✔. Chiều cao `1 + max(left, right)` đúng theo định nghĩa ✔.

Áp dụng cho `root` ⇒ `best` là đường kính.

## 12. Độ phức tạp

Gọi `n` = số node, `h` = chiều cao cây.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Baseline gọi `height` lại | `O(n²)` xấu nhất (`O(n log n)` nếu cân bằng) | `O(h)` | dễ hiểu, chậm với cây lệch |
| 2. DFS một lần (đệ quy) | `O(n)` | `O(h)` call stack | **khuyên dùng** |
| 3. DFS lặp bằng stack | `O(n)` | `O(n)` (dict `h` + stack) | không lo RecursionError |
| 4. Hai lần BFS | `O(n)` | `O(n)` | tổng quát cho cây bất kỳ |

`h` là `O(log n)` với cây cân bằng, `O(n)` với cây lệch.

## 13. Bộ test: mẫu đề, biên, phản ví dụ, ngẫu nhiên

In [20]:
tests = [
    ([1, 2, 3, 4, 5], 3),
    ([1, 2], 1),
    ([1], 0),
    ([1, None, 2, None, 3], 2),                                  # lệch phải
    ([1, 2, 3, 4, 5, None, None, 6, None, None, 7, 8, None, None, 9], 6),   # không qua root
    ([4, -7, -3, None, None, -9, -3, 9, -7, -4, None, 6, None, -6, -6, None, None, 0, 6, 5, None, 9, None, None, -1, -4, None, None, None, -2], 8),
]
methods = [diameter_naive, diameter_dfs, diameter_iterative, diameter_two_bfs]
for values, expected in tests:
    t = build_tree(values)
    for f in methods:
        got = f(t)
        assert got == expected, (f.__name__, values, got, expected)
    print(f"{str(values)[:45]:<46} -> {expected} ✔")
print("Tất cả test mẫu PASS!")

[1, 2, 3, 4, 5]                                -> 3 ✔
[1, 2]                                         -> 1 ✔
[1]                                            -> 0 ✔
[1, None, 2, None, 3]                          -> 2 ✔
[1, 2, 3, 4, 5, None, None, 6, None, None, 7,  -> 6 ✔
[4, -7, -3, None, None, -9, -3, 9, -7, -4, No  -> 8 ✔
Tất cả test mẫu PASS!


In [21]:
import random

def random_tree(n: int) -> Optional[TreeNode]:
    # Gắn node mới vào một vị trí trống ngẫu nhiên
    if n == 0:
        return None
    nodes = [TreeNode(0)]
    for v in range(1, n):
        while True:
            p = random.choice(nodes)
            side = random.choice(("left", "right"))
            if getattr(p, side) is None:
                child = TreeNode(v)
                setattr(p, side, child)
                nodes.append(child)
                break
    return nodes[0]

random.seed(543)
for _ in range(300):
    t = random_tree(random.randint(1, 40))
    answers = {f(t) for f in methods}
    assert len(answers) == 1, answers
print("300 cây ngẫu nhiên: 4 cách cho cùng kết quả.")

300 cây ngẫu nhiên: 4 cách cho cùng kết quả.


`getattr(obj, "left")` đọc thuộc tính theo tên dạng chuỗi; `setattr(obj, "left", value)` gán thuộc tính theo tên. Dùng để chọn ngẫu nhiên trái/phải mà không viết `if`.

## 14. Lỗi người mới hay gặp

### Lỗi 1: Chỉ tính đường qua root

In [22]:
def wrong_only_root(root):
    return height(root.left) + height(root.right)

print("sai:", wrong_only_root(skew), "| đúng:", diameter_dfs(skew))

sai: 5 | đúng: 6


Vì sao sai: đường kính có thể nằm hoàn toàn trong một cây con. Sửa: cập nhật `best` tại **mọi** node.

### Lỗi 2: Đếm số node thay vì số cạnh

Viết `best = max(best, left + right + 1)` sẽ ra số **node** trên đường, lớn hơn đáp án 1 đơn vị.

In [23]:
def wrong_count_nodes(root):
    best = 0
    def depth(node):
        nonlocal best
        if node is None:
            return 0
        l, r = depth(node.left), depth(node.right)
        best = max(best, l + r + 1)     # SAI: +1 là đếm node
        return 1 + max(l, r)
    depth(root)
    return best

print("sai:", wrong_count_nodes(build_tree([1, 2, 3, 4, 5])), "| đúng:", diameter_dfs(build_tree([1, 2, 3, 4, 5])))

sai: 4 | đúng: 3


### Lỗi 3: Trả về `best` thay vì chiều cao cho cha

Hàm đệ quy phải trả về **chiều cao** (`1 + max(left, right)`), không phải đáp án. Nếu trả nhầm `left + right` thì cha nhận một con số không phải chiều cao → sai.

### Lỗi 4: Quên `nonlocal`

Gây `UnboundLocalError` (xem mục 5.1). Sửa: thêm `nonlocal best` hoặc dùng `self.best`.

### Lỗi 5: Dùng biến toàn cục mà không reset giữa các test

Nếu dùng `self.best` hoặc biến global, nhớ gán lại 0 **đầu mỗi lần gọi** hàm chính; LeetCode có thể tái sử dụng cùng object `Solution` cho nhiều test.

## 15. Mẹo nhận dạng pattern và biến thể

### Mẹo
- Đề hỏi về **đường đi giữa hai node bất kỳ** trên cây → nghĩ "mỗi đường có một đỉnh", hàm đệ quy **trả về nhánh tốt nhất đi xuống**, còn đáp án gom **tại đỉnh** = trái + phải.
- Thấy baseline gọi lại hàm con (như `height`) nhiều lần → gộp vào **một lần post-order**.

### Biến thể
1. **LeetCode 124 – Binary Tree Maximum Path Sum** (Hard): như bài này nhưng cộng giá trị node, nhánh âm thì bỏ (`max(0, ...)`).
2. **LeetCode 687 – Longest Univalue Path**: chỉ nối nhánh khi giá trị con bằng giá trị cha.
3. **LeetCode 1522 – Diameter of N-Ary Tree**: lấy **hai** nhánh con sâu nhất.
4. **LeetCode 104 – Maximum Depth of Binary Tree**: chính là hàm `height`.

In [24]:
# Biến thể 2 demo: Longest Univalue Path (số cạnh)
def longest_univalue_path(root):
    best = 0
    def arrow(node):
        nonlocal best
        if node is None:
            return 0
        l, r = arrow(node.left), arrow(node.right)
        l = l + 1 if node.left and node.left.val == node.val else 0
        r = r + 1 if node.right and node.right.val == node.val else 0
        best = max(best, l + r)
        return max(l, r)
    arrow(root)
    return best

print(longest_univalue_path(build_tree([5, 4, 5, 1, 1, None, 5])))  # 2
print(longest_univalue_path(build_tree([1, 4, 5, 4, 4, None, 5])))  # 2

2
2


Ở đây `arrow` trả về số **cạnh** của nhánh cùng giá trị đi xuống — khác `depth` (số node) — nên công thức cập nhật là `l + r` sau khi đã cộng 1 cho mỗi cạnh nối được.

## 16. Bài tập tự luyện

### Bài 1 (dễ)
Viết `count_edges(root)` trả về số cạnh của cây (cây rỗng → 0).

In [25]:
def count_edges(root) -> int:
    # TODO
    pass

# assert count_edges(build_tree([1, 2, 3, 4, 5])) == 4
# assert count_edges(build_tree([1])) == 0

### Bài 2 (dễ)
Viết `longest_root_to_leaf_edges(root)`: số cạnh của đường dài nhất từ root xuống một lá.

In [26]:
def longest_root_to_leaf_edges(root) -> int:
    # TODO
    pass

# assert longest_root_to_leaf_edges(build_tree([1, 2, 3, 4, 5])) == 2
# assert longest_root_to_leaf_edges(build_tree([1])) == 0

### Bài 3 (vừa)
Viết `diameter_ex(root)` theo cách DFS một lần, **không dùng `nonlocal`** (gợi ý: cho hàm đệ quy trả về cặp `(chiều cao, đường kính trong cây con)`).

In [27]:
def diameter_ex(root) -> int:
    # TODO
    pass

# assert diameter_ex(build_tree([1, 2, 3, 4, 5])) == 3
# assert diameter_ex(skew) == 6

### Bài 4 (vừa)
Viết `diameter_endpoints(root)` trả về **cặp giá trị** của hai node đầu mút của một đường kính (bất kỳ cặp nào đúng).

In [28]:
def diameter_endpoints(root):
    # TODO
    pass

# a, b = diameter_endpoints(build_tree([1, 2, 3, 4, 5]))
# assert {a, b} in ({4, 3}, {5, 3})

### Bài 5 (khó hơn)
Viết `max_path_sum(root)` (LeetCode 124): đường đi có tổng giá trị node lớn nhất (đường có ít nhất 1 node).

In [29]:
def max_path_sum(root) -> int:
    # TODO
    pass

# assert max_path_sum(build_tree([1, 2, 3])) == 6
# assert max_path_sum(build_tree([-10, 9, 20, None, None, 15, 7])) == 42
# assert max_path_sum(build_tree([-3])) == -3

## 17. Gợi ý 3 mức

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Cây `n` node có bao nhiêu cạnh? | Đếm node bằng đệ quy | `max(0, n - 1)` |
| 2 | Liên hệ với `height` | Số cạnh = số node - 1 | `max(0, height(root) - 1)` |
| 3 | Trả về tuple | `(h, d)` của trái và phải | `d = max(dl, dr, hl + hr)` |
| 4 | Ngoài chiều cao, trả thêm **lá sâu nhất** | Tại đỉnh tốt nhất ghi lại hai lá | Con rỗng thì đầu mút là chính node |
| 5 | Nhánh trả cho cha chỉ đi một phía | Nhánh âm thì coi là 0 | `best = max(best, val + l + r)`, trả `val + max(l, r)` |

## 18. Đáp án tham khảo và tổng kết

In [30]:
def count_nodes(node):
    return 0 if node is None else 1 + count_nodes(node.left) + count_nodes(node.right)

def answer_count_edges(root) -> int:
    return max(0, count_nodes(root) - 1)

def answer_longest_root_to_leaf_edges(root) -> int:
    return max(0, height(root) - 1)

def answer_diameter_ex(root) -> int:
    def solve(node):
        if node is None:
            return 0, 0                       # (chiều cao, đường kính)
        hl, dl = solve(node.left)
        hr, dr = solve(node.right)
        return 1 + max(hl, hr), max(dl, dr, hl + hr)
    return solve(root)[1]

def answer_diameter_endpoints(root):
    best = [-1, None, None]
    def deepest(node):
        # trả về (chiều cao, giá trị lá sâu nhất)
        if node is None:
            return 0, None
        hl, leaf_l = deepest(node.left)
        hr, leaf_r = deepest(node.right)
        a = leaf_l if leaf_l is not None else node.val
        b2 = leaf_r if leaf_r is not None else node.val
        if hl + hr > best[0]:
            best[0], best[1], best[2] = hl + hr, a, b2
        return (1 + hl, a) if hl >= hr else (1 + hr, b2)
    deepest(root)
    return best[1], best[2]

def answer_max_path_sum(root) -> int:
    best = float("-inf")
    def gain(node):
        nonlocal best
        if node is None:
            return 0
        l = max(0, gain(node.left))
        r = max(0, gain(node.right))
        best = max(best, node.val + l + r)
        return node.val + max(l, r)
    gain(root)
    return best

assert answer_count_edges(build_tree([1, 2, 3, 4, 5])) == 4
assert answer_count_edges(build_tree([1])) == 0
assert answer_longest_root_to_leaf_edges(build_tree([1, 2, 3, 4, 5])) == 2
assert answer_longest_root_to_leaf_edges(build_tree([1])) == 0
assert answer_diameter_ex(build_tree([1, 2, 3, 4, 5])) == 3
assert answer_diameter_ex(skew) == 6
a, b2 = answer_diameter_endpoints(build_tree([1, 2, 3, 4, 5]))
assert {a, b2} in ({4, 3}, {5, 3})
print("Đầu mút đường kính của skew:", answer_diameter_endpoints(skew))
assert set(answer_diameter_endpoints(skew)) == {8, 9}
assert answer_max_path_sum(build_tree([1, 2, 3])) == 6
assert answer_max_path_sum(build_tree([-10, 9, 20, None, None, 15, 7])) == 42
assert answer_max_path_sum(build_tree([-3])) == -3
for _ in range(100):
    t = random_tree(random.randint(1, 30))
    assert answer_diameter_ex(t) == diameter_dfs(t)
print("Đáp án chạy PASS toàn bộ test!")

Đầu mút đường kính của skew: (8, 9)
Đáp án chạy PASS toàn bộ test!


### Template nhớ nhanh

```python
class Solution:
    def diameterOfBinaryTree(self, root: Optional[TreeNode]) -> int:
        best = 0
        def depth(node):
            nonlocal best
            if not node:
                return 0
            l, r = depth(node.left), depth(node.right)
            best = max(best, l + r)       # đường qua đỉnh node (số cạnh)
            return 1 + max(l, r)          # chiều cao trả cho cha
        depth(root)
        return best
```

### Checklist trước khi submit
- [ ] Cập nhật `best` tại **mọi** node, không chỉ root.
- [ ] `best` dùng `left + right` (số cạnh), **không** `+ 1`.
- [ ] Hàm đệ quy trả về **chiều cao** `1 + max(l, r)`, không trả `best`.
- [ ] Có `nonlocal best` (hoặc `self.best` được reset mỗi lần gọi).
- [ ] Đã nghĩ tới cây lệch sâu `10^4` → `sys.setrecursionlimit` hoặc bản lặp.
- [ ] Đã thử cây 1 node (đáp án 0) và cây có đường kính không qua root.